<h1 style="text-align: center; font-size: 50px;"> Run Workflow </h1>

# Notebook Overview
- Configure the Environment
- Define Constants and Paths
- Load Configuratons and Secrets
- Extract Markdown Files with Placeholders
- Parse Markdown Files
- Chunk Markdown Content
- Initialize Model
- Invoke Model on Each Chunk
- Save Results
- Log Execution Time

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.9 requires langchain-core>=1.0.0, but you have langchain-core 0.3.76 which is incompatible.
langchain-aws 1.0.0a1 requires langchain-core<2.0.0,>=1.0.0a4, but you have langchain-core 0.3.76 which is incompatible.
langchain-classic 1.0.0a1 requires langchain-core<2.0.0,>=1.0.0a7, but you have langchain-core 0.3.76 which is incompatible.
langchain-classic 1.0.0a1 requires langchain-text-splitters<2.0.0,>=1.0.0a1, but you have langchain-text-splitters 0.3.11 which is incompatible.
langchain-openai 1.0.0a4 requires langchain-core<2.0.0,>=1.0.0a7, but you have langchain-core 0.3.76 which is incompatible.
grpcio-status 1.80.0 requires protobuf<7.0.0,>=6.31.1, but you have protobuf 5.29.5 which is incompatible.
notebook 7.5.5 requires jupyterlab<4.6,>=4.5.6, but you have jupyterlab 4.2.7 which is i

In [2]:
MIN_TOTAL_RAM_GB = 16
MIN_TOTAL_VRAM_GB = 8


from ai_studio_blueprint_kit.memory_guard import run_memory_check_notebook


run_memory_check_notebook(
    min_total_ram_gb=MIN_TOTAL_RAM_GB,
    min_total_vram_gb=MIN_TOTAL_VRAM_GB,
)

## Step 0: Configure the Environment

In [3]:
# Standard library imports
import os  # Provides OS-related utilities
import sys  # Allows manipulation of Python runtime environment
import time  # Enables time-based operations

# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

# Local application-specific imports
from src.utils import logger

In [4]:
start_time = time.time()  

logger.info("Notebook execution started.")

In [5]:
# Standard Libraries
import os
import sys
import difflib
import json
import re
import yaml
from datetime import datetime
from collections import defaultdict
from typing import List
from pathlib import Path


# Internal Modules
from src.github_extractor import GitHubMarkdownProcessor
from src.utils import load_config_and_secrets, initialize_llm
from src.parser import parse_md_for_grammar_correction, restore_placeholders
from src.chunker import chunk_markdown
from src.prompt_templates import get_markdown_correction_prompt

# Other modules
import mlflow
from mlflow.models import evaluate

from IPython import get_ipython

In [6]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Define Constants and Paths

In [7]:
CONFIG_PATH = Path("../configs/config.yaml")
SECRETS_PATH = Path("../configs/secrets.yaml")

### Load Configurations and Secrets

In [8]:
from src.utils import load_config_and_secrets

# Load configuration and secrets
config, secrets = load_config_and_secrets(str(CONFIG_PATH), str(SECRETS_PATH) if SECRETS_PATH.exists() else None)
model_path = config.get("model_path")

# Also check for GitHub token in environment
github_token = os.getenv("AIS_GITHUB_ACCESS_TOKEN")
if github_token:
    if secrets is None:
        secrets = {}
    secrets["AIS_GITHUB_ACCESS_TOKEN"] = github_token
    logger.info("Loaded GITHUB_ACCESS_TOKEN from environment variable.")

# Validate model path if provided
if model_path and os.path.exists(model_path):
    logger.info(f"Model file found at: {model_path}")
else:
    logger.warning(f"Model file not found at: {model_path}")

## Step 1: Extracting and Parsing Markdown Files From GitHub Repositories

### Extract Markdown Files

In [9]:
# Define repo URL and GitHub access token
repo_url = "https://github.com/hp-david/test/tree/main" #"https://github.com/HPInc/AI-Blueprints" 
access_token = secrets.get("AIS_GITHUB_ACCESS_TOKEN")

# Create processor instance
processor = GitHubMarkdownProcessor(repo_url=repo_url, access_token=access_token)

# Run preprocessing workflow
markdowns = processor.run()

### Parse Markdown Files with Placeholders

In [10]:
parsed_markdowns = {}
placeholder_maps = {}

for filename, content in markdowns.items():
    # Parse the content and get placeholder map
    placeholder_map, processed_content = parse_md_for_grammar_correction(content)
    
    # Store the processed content (maintains dictionary structure for chunker)
    parsed_markdowns[filename] = processed_content
    
    # Store the placeholder map for restoration
    placeholder_maps[filename] = placeholder_map

logger.info(f"Parsed {len(parsed_markdowns)} files successfully")

### Chunk Markdown Content

In [11]:
all_chunks = {}  

# Chunk each file's content and store the results in a dictionary
for file_name, content in parsed_markdowns.items():
    chunks = chunk_markdown(content)
    all_chunks[file_name] = chunks

### Display Chunks (Optional)

In [12]:
'''
for file_name, chunks in all_chunks.items():
    logger.info(f"\n===== {file_name} =====\n")
    for i, chunk in enumerate(chunks):
        logger.info(f"\n--- Chunk {i+1} ---\n")
        logger.info(chunk)
        logger.info("\n" + "-" * 40 + "\n")
'''

'\nfor file_name, chunks in all_chunks.items():\n    logger.info(f"\n===== {file_name} =====\n")\n    for i, chunk in enumerate(chunks):\n        logger.info(f"\n--- Chunk {i+1} ---\n")\n        logger.info(chunk)\n        logger.info("\n" + "-" * 40 + "\n")\n'

## Step 2: Correct Markdown Files with LLM

In [13]:
# Get markdown correction prompt from prompt_templates module
correction_prompt = get_markdown_correction_prompt()

### Initialize Mode

In [14]:
%%time

# Set the model source from the configs
model_source = config.get("model_source", "local")

# Initialize llm using model_path from config
llm = initialize_llm(model_source, secrets, local_model_path=model_path)

# Create the LLM chain with the correction prompt
llm_chain = correction_prompt | llm

/opt/conda/lib/python3.12/site-packages/IPython/core/magics/execution.py:1412: UserWarning: WARNING! low_vram is not default parameter.
                low_vram was transferred to model_kwargs.
                Please confirm that low_vram is what you intended.
  exec(code, glob, local_ns)
/opt/conda/lib/python3.12/site-packages/IPython/core/magics/execution.py:1412: UserWarning: WARNING! rope_scaling is not default parameter.
                rope_scaling was transferred to model_kwargs.
                Please confirm that rope_scaling is what you intended.
  exec(code, glob, local_ns)
/opt/conda/lib/python3.12/site-packages/IPython/core/magics/execution.py:1412: UserWarning: WARNING! num_threads is not default parameter.
                num_threads was transferred to model_kwargs.
                Please confirm that num_threads is what you intended.
  exec(code, glob, local_ns)
llama_context: n_ctx_per_seq (32000) < n_ctx_train (131072) -- the full capacity of the model will not be uti

CPU times: user 1.23 s, sys: 2.86 s, total: 4.09 s
Wall time: 1min 38s


### Invoke Model on Each Chunk

In [15]:
%%time

results = []
count = 0

# Process each chunk through the language model and store the results
for file_name, chunks in all_chunks.items():  
    for chunk in chunks:
        # Send the chunks to the llm for correction
        response = llm_chain.invoke({"markdown": chunk})

        # Store the file name, original text, and corrected text
        results.append({
            "file": file_name,
            "original": chunk,
            "corrected": response
        })

        # Log progress (optional)
        logger.info(f"chunk {count} done")
        count += 1

CPU times: user 6min 56s, sys: 57.3 s, total: 7min 53s
Wall time: 7min 53s


### Display Corrected Chunks (Optional)

In [16]:
'''
for result in results:
    original_text = result["original"]
    corrected_text = result["corrected"]
    
    original_tokens = len(llm.client.tokenize(original_text.encode("utf-8")))
    corrected_tokens = len(llm.client.tokenize(corrected_text.encode("utf-8")))

    logger.info(f"\n===== {result['file']} =====\n")
    logger.info(f"--- Original ({original_tokens} tokens) ---\n")
    logger.info(original_text)
    logger.info(f"\n--- Corrected ({corrected_tokens} tokens) ---\n")
    logger.info(corrected_text)
    logger.info("\n" + "=" * 60 + "\n")
'''

'\nfor result in results:\n    original_text = result["original"]\n    corrected_text = result["corrected"]\n\n    original_tokens = len(llm.client.tokenize(original_text.encode("utf-8")))\n    corrected_tokens = len(llm.client.tokenize(corrected_text.encode("utf-8")))\n\n    logger.info(f"\n===== {result[\'file\']} =====\n")\n    logger.info(f"--- Original ({original_tokens} tokens) ---\n")\n    logger.info(original_text)\n    logger.info(f"\n--- Corrected ({corrected_tokens} tokens) ---\n")\n    logger.info(corrected_text)\n    logger.info("\n" + "=" * 60 + "\n")\n'

### Save Results

#### Save Raw Corrrected Markdowns

In [17]:
# Helper: Safe chunk joiner.
def safe_join_chunks(chunks: List[str]) -> str:
    """Rejoins a list of text chunks into a single string, preserving formatting and sentence boundaries.

    Ensures that chunks split mid-sentence get a space inserted appropriately.

    Args:
        chunks (List[str]): A list of processed text segments.

    Returns:
        str: The reassembled markdown text.
    """
    joined = ""
    for i, chunk in enumerate(chunks):
        if i == 0:
            joined += chunk
        else:
            prev = chunks[i - 1].rstrip()
            curr = chunk

            # Heuristic: Detect if a sentence was split across two chunks.
            if prev.endswith('.') and re.match(r'^[A-Z\"]', curr.lstrip()):
                # If it's a sentence break, add a single space to separate them.
                joined += ' ' + curr.lstrip()
            else:
                # Otherwise, join the chunk directly.
                joined += curr  
    return joined


# Group corrected chunks by file
corrected_chunks_by_file = defaultdict(list)
for result in results:
    corrected_chunks_by_file[result["file"]].append(result["corrected"])

# Rebuild each file from its corrected chunks with smart joining
rebuilt_corrected_files = {
    file_name: safe_join_chunks(chunks)
    for file_name, chunks in corrected_chunks_by_file.items()
}

# Create output directory
output_dir = Path("corrected")
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Restore placeholders and write final output
for file_name, corrected_content in rebuilt_corrected_files.items():
    placeholder_map = placeholder_maps.get(file_name, {})
    restored_content = restore_placeholders(corrected_content, placeholder_map)

    file_path = Path(file_name)

    # Construct the output path while preserving the original directory structure.
    output_path = output_dir / file_path.parent / f"{file_path.stem}_{timestamp}{file_path.suffix}"

    # This line is now crucial for creating the subdirectories.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(restored_content)

#### Save Corrected Markdowns with Diff

In [18]:
# Cell 2 (Corrected)

diff_output_dir = Path("corrected-diffs")
diff_output_dir.mkdir(parents=True, exist_ok=True)

for file_name, corrected_content in rebuilt_corrected_files.items():
    placeholder_map = placeholder_maps.get(file_name, {})
    restored_content = restore_placeholders(corrected_content, placeholder_map)

    # Get original content from markdowns dict
    original_content = markdowns.get(file_name)
    if original_content is None:
        logger.info(f"Warning: No original content for file {file_name}")
        continue

    # Create unified diff view (HTML side-by-side)
    differ = difflib.HtmlDiff(tabsize=4, wrapcolumn=80)
    diff_html = differ.make_file(
        original_content.splitlines(),
        restored_content.splitlines(),
        fromdesc=f"Original: {file_name}",
        todesc=f"Corrected: {file_name}",
        context=True,
        numlines=3
    )

    file_path = Path(file_name)

    # Construct the diff path while preserving the original directory structure.
    diff_path = diff_output_dir / file_path.parent / f"{file_path.stem}_{timestamp}{file_path.suffix}.html"
    
    # Ensure the subdirectories exist in the diffs folder too.
    diff_path.parent.mkdir(parents=True, exist_ok=True)
    
    with open(diff_path, "w", encoding="utf-8") as f:
        f.write(diff_html)

#### Save Chunks Into a JSON for Evaluation

In [19]:
results_path = Path("results.json")
with results_path.open("w", encoding="utf-8") as f:
    json.dump(results, f)

### Log Execution Time

In [20]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")

In [1]:
status = "Notebook execution completed successfully"
print(f"Message: {status}")

Message: Notebook execution completed successfully


In [ ]:
app = get_ipython()
app.kernel.do_shutdown(restart=False)

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).